<a href="https://colab.research.google.com/github/c5063565-coder/dnnls_final_project/blob/main/Experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.transforms.functional as FT
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import math
import os
import time
import json
import textwrap
import re

from bs4 import BeautifulSoup
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import BertTokenizer
from google.colab import drive
import gc

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

Device: cuda


In [2]:
drive.mount('/content/gdrive')

BASE_FOLDER = '/content/gdrive/MyDrive/DNN/DL_Checkpoints'

FOLDERS = {
    'base':BASE_FOLDER,
    'checkpoints': os.path.join(BASE_FOLDER, 'checkpoints'),
    'baseline':os.path.join(BASE_FOLDER, 'experiments', 'baseline'),
    'exp1':os.path.join(BASE_FOLDER, 'experiments', 'exp1_transformer_resnet'),
    'exp2':os.path.join(BASE_FOLDER, 'experiments', 'exp2_contrastive'),
}

for name, path in FOLDERS.items():
    os.makedirs(path, exist_ok=True)

print("Drive folder structure:")
for name, path in FOLDERS.items():
    print(f"  {name:12} → {path}")

def save_checkpoint(model, optimizer, epoch, loss, filename):
    path = os.path.join(FOLDERS['checkpoints'], filename)
    torch.save({
        'epoch':epoch,
        'model_state_dict':model.state_dict(),
        'optimizer_state_dict':optimizer.state_dict() if optimizer else None,
        'loss':loss,
    }, path)
    print(f"Checkpoint saved: checkpoints/{filename}")

def load_checkpoint(model, optimizer=None, filename="checkpoint.pth"):
    path = os.path.join(FOLDERS['checkpoints'], filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Not found: {path}")
    ckpt = torch.load(path, map_location=torch.device('cpu'))
    model.load_state_dict(ckpt['model_state_dict'])
    if optimizer and ckpt.get('optimizer_state_dict'):
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    print(f"Loaded: {filename}  (epoch {ckpt.get('epoch', 0)})")
    return model, optimizer, ckpt.get('epoch', 0), ckpt.get('loss', None)

def save_log(lines, experiment):
    filename = f"training_log_{experiment}.txt"
    path     = os.path.join(FOLDERS[experiment], filename)
    content  = '\n'.join(lines)
    with open(path, 'w') as f:       # save to Drive
        f.write(content)
    with open(filename, 'w') as f:   # save locally in Colab as backup
        f.write(content)

    print(f"Log saved → experiments/{experiment}/{filename}")
def save_figure(fig, filename, experiment):
    path = os.path.join(FOLDERS[experiment], filename)
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f"Figure saved → experiments/{experiment}/{filename}")
print("\nAll utilities ready.")


Mounted at /content/gdrive
Drive folder structure:
  base         → /content/gdrive/MyDrive/DNN/DL_Checkpoints
  checkpoints  → /content/gdrive/MyDrive/DNN/DL_Checkpoints/checkpoints
  baseline     → /content/gdrive/MyDrive/DNN/DL_Checkpoints/experiments/baseline
  exp1         → /content/gdrive/MyDrive/DNN/DL_Checkpoints/experiments/exp1_transformer_resnet
  exp2         → /content/gdrive/MyDrive/DNN/DL_Checkpoints/experiments/exp2_contrastive

All utilities ready.
